# 04 - Evaluación del Modelo y Auditoría de Equidad (Fairness)
Este notebook evalúa el modelo híbrido multimodal (TensorFlow/Keras + SentenceTransformer) y realiza una auditoría de equidad por sector económico.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.hybrid_model import HybridModel
from src.preprocessing.text.nlp_cleaner import NLPCleaner
from src.preprocessing.text.text_encoder import TextEncoder
from src.data_loader.data_splitter import DataSplitter
from src.evaluation.itaca_model_evaluator import ITACAModelEvaluator
from src.utils.constants import SPLITS_DIR, MODEL_KERAS_PATH, TABULAR_PREPROCESSOR_PATH, LABEL_ENCODER_PATH

%matplotlib inline
sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'src'

## 1. Carga de Artefactos y Datos de Prueba (Test)

In [ ]:
# Cargar artefactos
model = HybridModel.load(str(MODEL_KERAS_PATH))
tabular_preprocessor = joblib.load(TABULAR_PREPROCESSOR_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)

# Cargar split de prueba
test_df = pd.read_csv(SPLITS_DIR / 'test.csv', encoding='utf-8')
print(f"Registros en Test: {len(test_df)}")

## 2. Inferencia en el Conjunto de Test

In [ ]:
splitter = DataSplitter()
X_tab_df, X_text_ser, y_ser = splitter.split_features_and_target(test_df)

y_test_idx = label_encoder.transform(y_ser)
X_tab = tabular_preprocessor.transform(X_tab_df)

cleaner = NLPCleaner()
clean_texts = cleaner.clean_series(X_text_ser)
encoder = TextEncoder()
X_text = encoder.encode(clean_texts)

y_probs = model.predict({"tabular_input": X_tab, "text_input": X_text}, verbose=0)
y_preds = np.argmax(y_probs, axis=1)

## 3. Evaluación de Métricas y Auditoría de Equidad (Fairness) por Sector

In [ ]:
evaluator = ITACAModelEvaluator(max_fairness_delta=0.05, class_names=list(label_encoder.classes_))
results = evaluator.evaluate_and_audit(
    y_true_indices=y_test_idx,
    y_pred_indices=y_preds,
    y_probs=y_probs,
    sectores=test_df['sector'].values
)

print(json.dumps(results['global_metrics'], indent=4, ensure_ascii=False))

## 4. Visualizaciones de Matriz de Confusión y Disparidad por Sector

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de Confusión
cm = np.array(results['global_metrics']['confusion_matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
axes[0].set_title('Matriz de Confusión (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')

# Equidad por Sector
f1_sec = results['fairness_audit']['f1_by_sector']
sec_names = list(f1_sec.keys())
sec_scores = list(f1_sec.values())

axes[1].bar(sec_names, sec_scores, color='#3b82f6', width=0.5)
axes[1].axhline(np.mean(sec_scores), color='black', linestyle='--', label=f'Promedio ({np.mean(sec_scores):.4f})')
axes[1].set_ylim(0.0, 1.0)
axes[1].set_title('F1-Score Macro por Sector Económico', fontweight='bold')
axes[1].set_ylabel('F1-Macro')
axes[1].legend()

plt.tight_layout()
plt.show()